## Load CSV data

In [1]:
import pandas as pd

# Load the raw CSV file
df = pd.read_csv('../data/raw_reviews.csv')

print(df.shape)
df.head()

(4000, 5)


,review_text,rating,date,verified,country
0,✅Trip Verified| I'm not entirely sure as to wh...,8.0,2026-05-02,True,T Bayne(United Kingdom)2nd May 2026
1,✅Trip Verified| Our first time in the new bu...,8.0,2026-04-21,True,Kevin Tunnicliffe(United Kingdom)21st April 2026
2,✅Trip Verified| Highly commendable again on ...,10.0,2026-04-09,True,Brian Crockford(United Kingdom)9th April 2026
3,✅Trip Verified| Highly commendable on all fr...,10.0,2026-04-09,True,Brian Crockford(United Kingdom)9th April 2026
4,Not Verified| Although the staff on board t...,2.0,2026-04-06,False,Sarah Harrison(United Kingdom)6th April 2026


## Drop missing values

In [2]:
# Drop the 5 rows where rating is missing
df = df.dropna(subset=['rating'])

# Convert rating from float (7.0) to integer (7) for cleaner labels
df['rating'] = df['rating'].astype(int)

print(f"Rows after cleaning: {len(df)}")

Rows after cleaning: 3995


# Clean the review text

In [4]:
import re

def clean_text(text):
    # Return empty string if review is missing
    if not isinstance(text, str):
        return ''
    
    # Remove "Trip Verified |" prefix that Skytrax adds to verified reviews
    text = re.sub(r'✓ Trip Verified \|', '', text)
    
    # Remove special characters — keep only letters, numbers and spaces
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    
    # Convert everything to lowercase so "Great" and "great" are treated the same
    text = text.lower()
    
    # Remove extra whitespace gaps left behind after cleaning
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply the cleaning function to every review
df['clean_text'] = df['review_text'].apply(clean_text)

# Check a before and after example
print("BEFORE:", df['review_text'].iloc[0][:200])
print("\nAFTER: ", df['clean_text'].iloc[0][:200])

BEFORE: ✅Trip Verified| I'm not entirely sure as to what they wish to achieve by reducing the seat pitch. I know it's a fairly short flight, but some of us are over 1.80mts tall. If you do online check-in at 

AFTER:  trip verified im not entirely sure as to what they wish to achieve by reducing the seat pitch i know its a fairly short flight but some of us are over 180mts tall if you do online checkin at vienna th


## Assign sentiment labels

In [5]:
# Label each review based on its star rating
def assign_sentiment(rating):
    if rating <= 3:
        return 'negative'  # clearly unhappy
    elif rating <= 6:
        return 'neutral'   # mixed experience
    else:
        return 'positive'  # satisfied customer

df['sentiment'] = df['rating'].apply(assign_sentiment)

print("Sentiment distribution:")
print(df['sentiment'].value_counts())

Sentiment distribution:
sentiment
negative    1882
positive    1393
neutral      720
Name: count, dtype: int64


## Remove empty reviews

In [6]:
# Some reviews may be empty after cleaning
df = df[df['clean_text'].str.len() > 20]

print(f"Rows after removing empty reviews: {len(df)}")

Rows after removing empty reviews: 3995




# Save the cleaned data

In [7]:
# Save cleaned data to a new file
df.to_csv('../data/cleaned_reviews.csv', index=False)

print(f"Saved {len(df)} cleaned reviews to data/cleaned_reviews.csv")
print("\nFinal columns:", df.columns.tolist())
print("\nSample cleaned row:")
print(df[['clean_text', 'rating', 'sentiment']].iloc[0])

Saved 3995 cleaned reviews to data/cleaned_reviews.csv

Final columns: ['review_text', 'rating', 'date', 'verified', 'country', 'clean_text', 'sentiment']

Sample cleaned row:
clean_text    trip verified im not entirely sure as to what ...
rating                                                        8
sentiment                                              positive
Name: 0, dtype: object
